# MLIR Tutorial

## MLIR 101

This part of the notebook builds IR below with the `mlir_laksa` Python bindings.

```mlir
func.func @sum(%A: memref<8xi32>) -> i32 {
  %c0 = arith.constant 0 : i32
  %r  = affine.for %i = 0 to 8
          iter_args(%acc = %c0) -> (i32) {
    %v = affine.load %A[%i] : memref<8xi32>
    %s = arith.addi %acc, %v : i32
    affine.yield %s : i32
  }
  return %r : i32
}
```

In [ ]:
from mlir_laksa.ir import (
    AffineMap,
    Context,
    InsertionPoint,
    IntegerType,
    Location,
    MemRefType,
    Module,
)
from mlir_laksa.dialects import affine, arith, func

### Context and location

Every MLIR object lives in a `Context`, and every op carries a `Location` that diagnostics point to. Here all ops get `loc(unknown)`.

In [ ]:
ctx = Context()
ctx.__enter__()

loc = Location.unknown()
loc.__enter__()

### Module

`Module.create()` makes an empty `builtin.module`, the top-level container of the IR.

An `InsertionPoint` decides where newly created ops go. Pointing it at the module body places the function created next inside the module.

In [ ]:
module = Module.create()
ip_module = InsertionPoint(module.body)
ip_module.__enter__()

### Types and affine maps

In [ ]:
i32 = IntegerType.get_signless(32)
buffer_type = MemRefType.get([8], i32)

index_map = AffineMap.get_identity(1)

### Build the `func.func` op

The function takes the buffer and returns the sum.

In [ ]:
func_op = func.FuncOp("sum", ([buffer_type], [i32]))
block = func_op.add_entry_block()
(arg0,) = block.arguments

ip_body = InsertionPoint(block)
ip_body.__enter__()

print(func_op)

### Build the `affine.for` op

`affine.for` is a loop with affine bounds, here the constants 0 and 8. `iter_args` makes it carry values across iterations: each one starts at the given operand, is available inside the body, and is updated by `affine.yield` at the end of every iteration. After the last iteration the carried value becomes the result of the loop op, which is the sum.

In [ ]:
# Initialize accumulator with constant
c0 = arith.ConstantOp(i32, 0)

loop = affine.AffineForOp(0, 8, iter_args=[c0.result])

with InsertionPoint(loop.body):
    i = loop.induction_variable
    (acc,) = loop.inner_iter_args
    v = affine.AffineLoadOp(i32, arg0, [i], index_map)
    s = arith.AddIOp(acc, v.result)
    affine.AffineYieldOp([s.result])

print(loop)

### Return

`func.return` hands the loop result back as the function result.

In [ ]:
func.ReturnOp([loop.results[0]])
ip_body.__exit__(None, None, None)

print(module)

## Exercise 1

Check how the operations are created and build the loop below in MLIR yourself ;)

```c
for (size_t i = 0; i < 8; i++)
    B[i] = A[i] * 2
```

In [ ]:
# New Module
module = Module.create()
ip_module = InsertionPoint(module.body)
ip_module.__enter__()

# New func
func_op = func.FuncOp("scale", ([buffer_type, buffer_type], []))
block = func_op.add_entry_block()
arg0, arg1 = block.arguments

ip_body = InsertionPoint(block)
ip_body.__enter__()

# Write your loops here!

c2 = arith.ConstantOp(i32, 2)

loop = affine.AffineForOp(0, 8)

with InsertionPoint(loop.body):
    i = loop.induction_variable
    v = affine.AffineLoadOp(i32, arg0, [i], index_map)
    m = arith.MulIOp(v.result, c2.result)
    affine.AffineStoreOp(m.result, arg1, [i], index_map)
    affine.AffineYieldOp([])

# Return
func.ReturnOp([])
ip_body.__exit__(None, None, None)
ip_module.__exit__(None, None, None)

print(module)

## Exercise 2

Get familiar with the `Pass` mechanism.

### What's different in `scf`

In [ ]:
from mlir_laksa.passmanager import PassManager

pm = PassManager(context=ctx)
pm.add("lower-affine")
pm.run(module.operation)

print(module)

### Whatabout `cf`

In [ ]:
from mlir_laksa.passmanager import PassManager

pm = PassManager(context=ctx)
pm.add("convert-scf-to-cf")
pm.run(module.operation)

print(module)

### What does it look like in `llvm`

In [ ]:
from mlir_laksa.passmanager import PassManager

pm = PassManager(context=ctx)
pm.add("convert-to-llvm")
pm.run(module.operation)

print(module)

## Exercise 3

Build the graph in the slides with the `dfg` dialect.

In [ ]:
from mlir_laksa.ir import FunctionType, TypeAttr
from mlir_laksa.dialects import dfg

In [ ]:
# New module
module = Module.create()
ip_module = InsertionPoint(module.body)
ip_module.__enter__()

# DFG Types
in_type = dfg.InputType.get(element_type=i32)
out_type = dfg.OutputType.get(element_type=i32)

# Build a node as Operator
node0 = dfg.OperatorOp("node0", TypeAttr.get(FunctionType.get([], [i32, i32])))
node1 = dfg.OperatorOp("node1", TypeAttr.get(FunctionType.get([i32], [i32])))
node2 = dfg.OperatorOp("node2", TypeAttr.get(FunctionType.get([i32], [i32])))
node3 = dfg.OperatorOp("node3", TypeAttr.get(FunctionType.get([i32, i32], [])))

# Build the graph as Region
graph = dfg.RegionOp("graph", TypeAttr.get(FunctionType.get([], [])))
block = graph.body.blocks.append()

ip_body = InsertionPoint(block)
ip_body.__enter__()

# Build channels to connect nodes
ch01_in, ch01_out = dfg.ChannelOp(in_type, out_type, i32).results
ch02_in, ch02_out = dfg.ChannelOp(in_type, out_type, i32).results
ch13_in, ch13_out = dfg.ChannelOp(in_type, out_type, i32).results
ch23_in, ch23_out = dfg.ChannelOp(in_type, out_type, i32).results

# Instantiate nodes with channels
dfg.InstantiateOp("node0", [], [ch01_in, ch02_in])
dfg.InstantiateOp("node1", [ch01_out], [ch13_in])
dfg.InstantiateOp("node2", [ch02_out], [ch23_in])
dfg.InstantiateOp("node3", [ch13_out, ch23_out], [])

ip_body.__exit__(None, None, None)
ip_module.__exit__(None, None, None)

print(module)

### To `dot` file

Check if the graph is what you expected to be

In [ ]:
import graphviz
from mlir_laksa.ir import UnitAttr

graph.operation.attributes["laksa.root"] = UnitAttr.get()

dot = dfg.translate_to_dot(module.operation)
print(dot)

graphviz.Source(dot)

## Exercise 4

Let's play with LAKSA and DSE.

### A quantized convolution layer

The layer reads a `1x32x32x8` int8 feature map in NHWC layout, convolves it with eight `3x3x8` filters (stride 1, no padding), and requantizes the `i32` accumulators back to int8 with a fused ReLU. The output is `1x30x30x8`.

It takes three `linalg` ops: a fill that initializes the accumulator, the convolution, and the requantization.

In [ ]:
import numpy as np
from mlir_laksa.ir import AffineExpr, DenseElementsAttr, IntegerAttr, RankedTensorType
from mlir_laksa.dialects import linalg, tensor

### Parameters and helpers

The layer has two parameter tensors, both drawn at random here:

* `WEIGHTS`: `8x3x3x8` int8 in FHWC layout (output channel, kernel row, kernel column, input channel), which is the filter layout `linalg.conv_2d_nhwc_fhwc_q` expects.
* `MULTIPLIERS`: one int32 per output channel. Together with a shift of 40, each one encodes that channel's requantization scale `multiplier / 2^40`.

Constant tensors are dense elements attributes in MLIR, e.g. `dense<[1, 2, 3]> : tensor<3xi32>`. `dense` builds one from a numpy array, `splat` builds one in which every element has the same value, and `constant` wraps an attribute in an `arith.constant` so that other ops can use it.

In [ ]:
rng = np.random.default_rng(0)
WEIGHTS = rng.integers(-127, 127, size=(8, 3, 3, 8), dtype=np.int8, endpoint=True)
MULTIPLIERS = rng.integers(1_280_000_000, 1_320_000_000, size=8, dtype=np.int32)

def dense(array, elem_type):
    return DenseElementsAttr.get(
        np.ascontiguousarray(array),
        type=RankedTensorType.get(list(array.shape), elem_type),
    )

def splat(shape, elem_type, value):
    return DenseElementsAttr.get_splat(
        RankedTensorType.get(shape, elem_type), IntegerAttr.get(elem_type, value)
    )

def constant(attr):
    return arith.ConstantOp(attr.type, attr)

### Types and affine maps

Input and output are int8, while the convolution accumulates in `i32`. `i64` only appears inside the requantization arithmetic.

A `linalg.generic` runs over the four output dimensions `(d0, d1, d2, d3)` = (N, H, W, C), and every operand has an affine map that picks its element at each point:

* `map_full` is the identity, for operands with the full output shape.
* `map_bias` is `(d0, d1, d2, d3) -> (d3)`, for per-channel operands such as the bias, the multipliers and the shifts, which are broadcast over all pixels.

In [ ]:
i8 = IntegerType.get_signless(8)
i32 = IntegerType.get_signless(32)
i64 = IntegerType.get_signless(64)

input_type = RankedTensorType.get([1, 32, 32, 8], i8)
output_type = RankedTensorType.get([1, 30, 30, 8], i8)
acc_type = RankedTensorType.get([1, 30, 30, 8], i32)

d0, d1, d2, d3 = (AffineExpr.get_dim(i) for i in range(4))
map_bias = AffineMap.get(4, 0, [d3])
map_full = AffineMap.get(4, 0, [d0, d1, d2, d3])

### Build the layer

One block builds the whole layer in a new module:

* `func.func @main` takes the input feature map and returns the output feature map.
* The constants come first: the zero points and clamp bounds `c_neg128`, `c0` and `c127`, the per-channel `multipliers` and `shifts`, the rounding constants `c_half`, `c_neg_half`, `c31` and `c1_i64`, the `weights` and an all-zero `bias`. They sit at the top of the function, because the body of a `linalg.generic` may use values defined outside of it.
* **Fill.** Linalg ops use destination-passing style: the inputs go into `ins`, and the initial value of the result goes into `outs`. `tensor.empty` provides that initial tensor as a pure shape without any data. The first `linalg.generic` initializes the `i32` accumulator with the bias of each channel.
* **Quantized convolution.** `linalg.conv_2d_nhwc_fhwc_q` accumulates `acc[n, oh, ow, f] += (x[n, oh+kh, ow+kw, c] - x_zp) * (w[f, kh, kw, c] - w_zp)` on top of the bias-filled accumulator, with `x_zp = -128` and `w_zp = 0`. A named op created from Python starts with an empty region, and `fill_builtin_region` fills in its body.
* **ReLU and requantization.** The last `linalg.generic` computes `out = clamp(((acc * multiplier + round) >> shift) + out_zp, -128, 127)` with `out_zp = -128`. The `cmpi`/`select` pairs implement the double rounding of TOSA's `rescale` op. With an output zero point of -128, the real value 0 is stored as -128, so clamping at -128 is exactly the ReLU.
* `func.return` hands the requantized tensor back as the function result.

In [ ]:
module = Module.create()
ip_module = InsertionPoint(module.body)
ip_module.__enter__()

func_op = func.FuncOp("main", ([input_type], [output_type]))
block = func_op.add_entry_block()
(arg0,) = block.arguments

ip_body = InsertionPoint(block)
ip_body.__enter__()

c127 = arith.ConstantOp(i32, 127)
c_neg_half = arith.ConstantOp(i64, -1073741824)
c_half = arith.ConstantOp(i64, 1073741824)
c31 = arith.ConstantOp(i32, 31)
c1_i64 = arith.ConstantOp(i64, 1)
shifts = constant(splat([8], i8, 40))
multipliers = constant(dense(MULTIPLIERS, i32))
c0 = arith.ConstantOp(i32, 0)
c_neg128 = arith.ConstantOp(i32, -128)
weights = constant(dense(WEIGHTS, i8))
bias = constant(splat([8], i32, 0))

acc_init = tensor.EmptyOp([1, 30, 30, 8], i32)

fill = linalg.GenericOp(
    [acc_type],
    [bias.result],
    [acc_init.result],
    [map_bias, map_full],
    ["parallel"] * 4,
)
fill_block = fill.regions[0].blocks.append(i32, i32)
with InsertionPoint(fill_block):
    in_, _out = fill_block.arguments
    linalg.YieldOp([in_])

conv = linalg.Conv2DNhwcFhwcQOp(
    [acc_type],
    [arg0, weights.result, c_neg128.result, c0.result],
    [fill.results[0]],
    strides=dense(np.array([1, 1], dtype=np.int64), i64),
    dilations=dense(np.array([1, 1], dtype=np.int64), i64),
)
linalg.fill_builtin_region(conv.operation)

out_init = tensor.EmptyOp([1, 30, 30, 8], i8)

relu = linalg.GenericOp(
    [output_type],
    [conv.results[0], multipliers.result, shifts.result],
    [out_init.result],
    [map_full, map_bias, map_bias, map_full],
    ["parallel"] * 4,
)
relu_block = relu.regions[0].blocks.append(i32, i32, i8, i8)
with InsertionPoint(relu_block):
    in_, in_mul, in_shift, _out = relu_block.arguments
    shift_i32 = arith.ExtUIOp(i32, in_shift)
    acc = arith.ExtSIOp(i64, in_)
    mul = arith.ExtSIOp(i64, in_mul)
    prod = arith.MulIOp(acc.result, mul.result)
    shift_i64 = arith.ExtUIOp(i64, in_shift)
    one_shl = arith.ShLIOp(c1_i64.result, shift_i64.result)
    round_ = arith.ShRUIOp(one_shl.result, c1_i64.result)
    rounded = arith.AddIOp(prod.result, round_.result)
    is_pos = arith.CmpIOp(arith.CmpIPredicate.sge, in_, c0.result)
    half = arith.SelectOp(is_pos.result, c_half.result, c_neg_half.result)
    biased = arith.AddIOp(half.result, rounded.result)
    wide = arith.CmpIOp(arith.CmpIPredicate.sgt, shift_i32.result, c31.result)
    picked = arith.SelectOp(wide.result, biased.result, rounded.result)
    shifted = arith.ShRSIOp(picked.result, shift_i64.result)
    trunc32 = arith.TruncIOp(i32, shifted.result)
    zp = arith.AddIOp(trunc32.result, c_neg128.result)
    clamped_lo = arith.MaxSIOp(zp.result, c_neg128.result)
    clamped = arith.MinSIOp(clamped_lo.result, c127.result)
    narrow = arith.TruncIOp(i8, clamped.result)
    linalg.YieldOp([narrow.result])

func.ReturnOp([relu.results[0]])
ip_body.__exit__(None, None, None)
ip_module.__exit__(None, None, None)

print(module)

### To `emithls`

`convert-to-emithls` lowers the `linalg` IR to LAKSA's `emithls` dialect for Vitis HLS. Its last step is a design-space exploration that places HLS pragmas under a resource budget of `num_bram` BRAM_18K blocks and `num_dsp` DSPs. The defaults are the resources of the Kria K26 on the KV260.

Change the number of resource budget, and see what happens.

In [ ]:
import mlir_laksa.conversion as conversion

num_bram = 288
num_dsp = 1248

layer_hls = Module.parse(str(module), context=ctx)

pm = PassManager(context=ctx)
conversion.add_convert_to_emithls_pipeline(pm, available_bram=num_bram, available_dsp=num_dsp)
pm.run(layer_hls.operation)

print(layer_hls)

### Build a residual block

In [ ]:
from mlir_laksa.ir import AffineConstantExpr, IndexType

rng = np.random.default_rng(1)
W1, W2, W3 = (
    rng.integers(-127, 127, size=(8, 3, 3, 8), dtype=np.int8, endpoint=True)
    for _ in range(3)
)
M1, M2, M3 = (
    rng.integers(1 << 30, (1 << 31) - 1, size=8, dtype=np.int32) for _ in range(3)
)

index = IndexType.get()
map_batch0 = AffineMap.get(4, 0, [AffineConstantExpr.get(0), d1, d2, d3])

x_type = RankedTensorType.get([1, 32, 32, 8], i8)
y_type = RankedTensorType.get([1, 28, 28, 8], i8)
acc30_type = RankedTensorType.get([1, 30, 30, 8], i32)
act30_type = RankedTensorType.get([1, 30, 30, 8], i8)
acc28_type = RankedTensorType.get([1, 28, 28, 8], i32)


def fill(init, acc_type):
    op = linalg.GenericOp(
        [acc_type], [bias.result], [init], [map_bias, map_full], ["parallel"] * 4
    )
    blk = op.regions[0].blocks.append(i32, i32)
    with InsertionPoint(blk):
        linalg.YieldOp([blk.arguments[0]])
    return op.results[0]


def conv(x, w, acc, acc_type):
    op = linalg.Conv2DNhwcFhwcQOp(
        [acc_type],
        [x, w.result, c_neg128.result, c0.result],
        [acc],
        strides=dense(np.array([1, 1], dtype=np.int64), i64),
        dilations=dense(np.array([1, 1], dtype=np.int64), i64),
    )
    linalg.fill_builtin_region(op.operation)
    return op.results[0]


def requant(acc, mul, shift, init, out_type):
    op = linalg.GenericOp(
        [out_type],
        [acc, mul.result, shift.result],
        [init],
        [map_full, map_bias, map_bias, map_full],
        ["parallel"] * 4,
    )
    blk = op.regions[0].blocks.append(i32, i32, i8, i8)
    with InsertionPoint(blk):
        in_, in_mul, in_shift, _out = blk.arguments
        shift_i32 = arith.ExtUIOp(i32, in_shift)
        acc_i64 = arith.ExtSIOp(i64, in_)
        mul_i64 = arith.ExtSIOp(i64, in_mul)
        prod = arith.MulIOp(acc_i64.result, mul_i64.result)
        shift_i64 = arith.ExtUIOp(i64, in_shift)
        one_shl = arith.ShLIOp(c1_i64.result, shift_i64.result)
        round_ = arith.ShRUIOp(one_shl.result, c1_i64.result)
        rounded = arith.AddIOp(prod.result, round_.result)
        is_pos = arith.CmpIOp(arith.CmpIPredicate.sge, in_, c0.result)
        half = arith.SelectOp(is_pos.result, c_half.result, c_neg_half.result)
        biased = arith.AddIOp(half.result, rounded.result)
        wide = arith.CmpIOp(arith.CmpIPredicate.sgt, shift_i32.result, c31.result)
        picked = arith.SelectOp(wide.result, biased.result, rounded.result)
        shifted = arith.ShRSIOp(picked.result, shift_i64.result)
        trunc32 = arith.TruncIOp(i32, shifted.result)
        zp = arith.AddIOp(trunc32.result, c_neg128.result)
        clamped_lo = arith.MaxSIOp(zp.result, c_neg128.result)
        clamped = arith.MinSIOp(clamped_lo.result, c127.result)
        narrow = arith.TruncIOp(i8, clamped.result)
        linalg.YieldOp([narrow.result])
    return op.results[0]


def widen(x, init, rnd, shift):
    op = linalg.GenericOp(
        [acc30_type], [x], [init], [map_full, map_full], ["parallel"] * 4
    )
    blk = op.regions[0].blocks.append(i8, i32)
    with InsertionPoint(blk):
        in_, _out = blk.arguments
        ext = arith.ExtSIOp(i32, in_)
        centered = arith.SubIOp(ext.result, c_neg128.result)
        wide = arith.ExtSIOp(i64, centered.result)
        scaled = arith.MulIOp(wide.result, c_half.result)
        rounded = arith.AddIOp(scaled.result, rnd.result)
        shifted = arith.ShRSIOp(rounded.result, shift.result)
        narrow = arith.TruncIOp(i32, shifted.result)
        linalg.YieldOp([narrow.result])
    return op.results[0]

In [ ]:
module = Module.create()
ip_module = InsertionPoint(module.body)
ip_module.__enter__()

func_op = func.FuncOp("main", ([x_type], [y_type]))
block = func_op.add_entry_block()
(arg0,) = block.arguments

ip_body = InsertionPoint(block)
ip_body.__enter__()

c_2p48 = arith.ConstantOp(i64, 281474976710656)
c49 = arith.ConstantOp(i64, 49)
c1024 = arith.ConstantOp(i64, 1024)
c11 = arith.ConstantOp(i64, 11)
c32 = arith.ConstantOp(i64, 32)
c_2p31 = arith.ConstantOp(i64, 2147483648)
c_skip_mul = arith.ConstantOp(i64, 1633833687)
c512 = arith.ConstantOp(i64, 512)
c10 = arith.ConstantOp(i64, 10)
shifts3 = constant(dense(np.array([39, 39, 39, 40, 40, 39, 40, 40], dtype=np.int8), i8))
multipliers3 = constant(dense(M3, i32))
shifts2 = constant(splat([8], i8, 40))
multipliers2 = constant(dense(M2, i32))
c127 = arith.ConstantOp(i32, 127)
c_neg_half = arith.ConstantOp(i64, -1073741824)
c_half = arith.ConstantOp(i64, 1073741824)
c31 = arith.ConstantOp(i32, 31)
c1_i64 = arith.ConstantOp(i64, 1)
shifts1 = constant(splat([8], i8, 41))
multipliers1 = constant(dense(M1, i32))
c_neg128_i8 = arith.ConstantOp(i8, -128)
c0 = arith.ConstantOp(i32, 0)
c_neg128 = arith.ConstantOp(i32, -128)
bias = constant(splat([8], i32, 0))
weights3 = constant(dense(W3, i8))
weights2 = constant(dense(W2, i8))
weights1 = constant(dense(W1, i8))

acc30_init = tensor.EmptyOp([1, 30, 30, 8], i32)
acc30 = fill(acc30_init.result, acc30_type)
conv1 = conv(arg0, weights1, acc30, acc30_type)
act30_init = tensor.EmptyOp([1, 30, 30, 8], i8)
act1 = requant(conv1, multipliers1, shifts1, act30_init.result, act30_type)

pad = tensor.PadOp(x_type, act1, [], [], [0, 1, 1, 0], [0, 1, 1, 0])
pad_block = pad.region.blocks.append(index, index, index, index)
with InsertionPoint(pad_block):
    tensor.YieldOp(c_neg128_i8.result)

conv2 = conv(pad.result, weights2, acc30, acc30_type)
act2 = requant(conv2, multipliers2, shifts2, act30_init.result, act30_type)

skip_wide = widen(act1, acc30_init.result, c512, c10)

skip_scaled = linalg.GenericOp(
    [acc30_type], [skip_wide], [acc30_init.result], [map_full, map_full], ["parallel"] * 4
)
skip_block = skip_scaled.regions[0].blocks.append(i32, i32)
with InsertionPoint(skip_block):
    in_, _out = skip_block.arguments
    wide = arith.ExtSIOp(i64, in_)
    prod = arith.MulIOp(wide.result, c_skip_mul.result)
    rounded = arith.AddIOp(prod.result, c_2p31.result)
    is_pos = arith.CmpIOp(arith.CmpIPredicate.sge, in_, c0.result)
    half = arith.SelectOp(is_pos.result, c_half.result, c_neg_half.result)
    biased = arith.AddIOp(half.result, rounded.result)
    shifted = arith.ShRUIOp(biased.result, c32.result)
    narrow = arith.TruncIOp(i32, shifted.result)
    linalg.YieldOp([narrow.result])

main_wide = widen(act2, acc30_init.result, c1024, c11)

summed = linalg.GenericOp(
    [acc30_type],
    [skip_scaled.results[0], main_wide],
    [acc30_init.result],
    [map_batch0, map_batch0, map_full],
    ["parallel"] * 4,
)
sum_block = summed.regions[0].blocks.append(i32, i32, i32)
with InsertionPoint(sum_block):
    lhs, rhs, _out = sum_block.arguments
    total = arith.AddIOp(lhs, rhs)
    linalg.YieldOp([total.result])

merged = linalg.GenericOp(
    [act30_type], [summed.results[0]], [act30_init.result], [map_full, map_full], ["parallel"] * 4
)
merged_block = merged.regions[0].blocks.append(i32, i8)
with InsertionPoint(merged_block):
    in_, _out = merged_block.arguments
    wide = arith.ExtSIOp(i64, in_)
    prod = arith.MulIOp(wide.result, c_half.result)
    rounded = arith.AddIOp(prod.result, c_2p48.result)
    is_pos = arith.CmpIOp(arith.CmpIPredicate.sge, in_, c0.result)
    half = arith.SelectOp(is_pos.result, c_half.result, c_neg_half.result)
    biased = arith.AddIOp(half.result, rounded.result)
    shifted = arith.ShRSIOp(biased.result, c49.result)
    trunc32 = arith.TruncIOp(i32, shifted.result)
    zp = arith.AddIOp(trunc32.result, c_neg128.result)
    clamped_lo = arith.MaxSIOp(zp.result, c_neg128.result)
    clamped = arith.MinSIOp(clamped_lo.result, c127.result)
    narrow = arith.TruncIOp(i8, clamped.result)
    linalg.YieldOp([narrow.result])

acc28_init = tensor.EmptyOp([1, 28, 28, 8], i32)
acc28 = fill(acc28_init.result, acc28_type)
conv3 = conv(merged.results[0], weights3, acc28, acc28_type)
act28_init = tensor.EmptyOp([1, 28, 28, 8], i8)
act3 = requant(conv3, multipliers3, shifts3, act28_init.result, y_type)

func.ReturnOp([act3])
ip_body.__exit__(None, None, None)
ip_module.__exit__(None, None, None)

print(module)

### To `emithls`

The same pipeline as above.

In [ ]:
num_bram = 288
num_dsp = 1248

residual_hls = Module.parse(str(module), context=ctx)

pm = PassManager(context=ctx)
conversion.add_convert_to_emithls_pipeline(pm, available_bram=num_bram, available_dsp=num_dsp)
pm.run(residual_hls.operation)

print(residual_hls)

### Check the generated files

There is a centeral compiler entry `ladle`, which takes care of "everything" for you ;)

In [ ]:
with open("input.mlir", "w") as f:
    f.write(str(module))

!ladle input.mlir --hls --num-bram={num_bram} --num-dsp={num_dsp} -o out_dir